In [4]:
import cv2
import numpy as np
import tensorflow as tf

print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

OpenCV: 4.10.0
NumPy: 1.23.5
TensorFlow: 2.10.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model


# Load Haar Cascade for face detection
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

# Check cascade
if face_cascade.empty():
    print("Error: Haar Cascade could not be loaded!")
    exit()

# Load trained emotion recognition model
model = load_model("second_model.keras")

# Emotion classes
classes = [
    "Angry",
    "Disgust",
    "Fear",
    "Happy",
    "Neutral",
    "Sad",
    "Surprise"
]


# Preprocessing function
def preprocess_face(face_img):
    # Resize to model input size
    face = cv2.resize(face_img, (48, 48))

    # Convert to float32
    # DO NOT divide by 255
    # Model already has Rescaling(1./255)
    face = face.astype("float32")

    # Add channel dimension: (48,48) -> (48,48,1)
    face = np.expand_dims(face, axis=-1)

    # Add batch dimension: (48,48,1) -> (1,48,48,1)
    face = np.expand_dims(face, axis=0)

    return face


# Emotion prediction
def predict_emotion(face_img):
    processed = preprocess_face(face_img)

    # Model prediction
    prediction = model.predict(processed, verbose=0)[0]

    # Get highest probability
    class_idx = np.argmax(prediction)

    # Show probabilities in terminal
    probabilities = {
        classes[i]: round(float(prediction[i]), 3)
        for i in range(len(classes))
    }

    print(probabilities)

    return classes[class_idx]


# Start webcam
cap = cv2.VideoCapture(1)

if not cap.isOpened():
    print("Error: Could not open webcam!")
    exit()


while True:

    ret, frame = cap.read()

    if not ret:
        print("Error: Could not read frame!")
        break

    # Convert frame to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=10,
        minSize=(48, 48)
    )

    # Process each detected face
    for (x, y, w, h) in faces:

        # Crop grayscale face
        face_gray = gray[y:y+h, x:x+w]

        # Predict emotion
        label = predict_emotion(face_gray)

        # Draw bounding box
        cv2.rectangle(
            frame,
            (x, y),
            (x + w, y + h),
            (0, 255, 0),
            2
        )

        # Draw emotion label
        cv2.putText(
            frame,
            label,
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0, 255, 0),
            2
        )

    # Show webcam
    cv2.imshow("Webcam Emotion Detection", frame)

    # Press Q to quit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# Cleanup
cap.release()
cv2.destroyAllWindows()

{'Angry': 0.037, 'Disgust': 0.0, 'Fear': 0.009, 'Happy': 0.004, 'Neutral': 0.916, 'Sad': 0.034, 'Surprise': 0.0}
{'Angry': 0.06, 'Disgust': 0.0, 'Fear': 0.015, 'Happy': 0.016, 'Neutral': 0.84, 'Sad': 0.069, 'Surprise': 0.0}
{'Angry': 0.075, 'Disgust': 0.0, 'Fear': 0.045, 'Happy': 0.004, 'Neutral': 0.69, 'Sad': 0.186, 'Surprise': 0.0}
{'Angry': 0.071, 'Disgust': 0.0, 'Fear': 0.028, 'Happy': 0.005, 'Neutral': 0.728, 'Sad': 0.168, 'Surprise': 0.0}
{'Angry': 0.129, 'Disgust': 0.0, 'Fear': 0.026, 'Happy': 0.022, 'Neutral': 0.696, 'Sad': 0.127, 'Surprise': 0.0}
{'Angry': 0.056, 'Disgust': 0.0, 'Fear': 0.012, 'Happy': 0.011, 'Neutral': 0.834, 'Sad': 0.087, 'Surprise': 0.0}
{'Angry': 0.08, 'Disgust': 0.0, 'Fear': 0.015, 'Happy': 0.01, 'Neutral': 0.799, 'Sad': 0.096, 'Surprise': 0.0}
{'Angry': 0.246, 'Disgust': 0.002, 'Fear': 0.085, 'Happy': 0.01, 'Neutral': 0.476, 'Sad': 0.155, 'Surprise': 0.026}
{'Angry': 0.068, 'Disgust': 0.0, 'Fear': 0.006, 'Happy': 0.002, 'Neutral': 0.782, 'Sad': 0.142, 'S